## Cell 1 — PUF Toolbox
Run this once per session before any other PUF cell.

In [ ]:
# install reedsolo if not already installed
import subprocess
subprocess.run(["pip", "install", "reedsolo", "--quiet"], check=True)

import reedsolo
import hashlib
import os
import json
import math
from pathlib import Path


def bits_to_bytes(bits):
    """Pack a bit array into bytes, MSB-first, zero-padded to multiple of 8."""
    bits = np.asarray(bits, np.uint8)
    pad  = (8 - len(bits) % 8) % 8
    bits = np.concatenate([bits, np.zeros(pad, np.uint8)])
    return bytes(int(''.join(map(str, bits[i:i+8])), 2)
                 for i in range(0, len(bits), 8))


def bytes_to_bits(b, n_bits=None):
    """Unpack bytes back into a bit array."""
    arr = np.array([int(bit) for byte in b
                    for bit in format(byte, '08b')], np.uint8)
    return arr[:n_bits] if n_bits else arr


def extract_puf_bits(a_dict, sensors, n_quant=4, rank_bits=None):
    """
    Extract PUF bits from sensor amplitudes {sensor: mV}.

    rank_bits : if provided, uses these directly (pass modal_bitstring
                from the Averaging cell to avoid recomputing the same
                pairwise comparisons that cell already computed).
                If None, recomputes from a_dict.

    ratio bits: each sensor's log-amplitude relative to the geometric
                mean, quantized to n_quant bits. New entropy not present
                in the Averaging cell output.

    Total bits for 4 sensors, n_quant=4:  6 rank + 16 ratio = 22 bits
    """
    # rank bits — reuse from Averaging cell if available
    if rank_bits is not None:
        rank = np.asarray(rank_bits, dtype=np.uint8)
    else:
        rank = np.array(
            [1 if a_dict[sensors[i]] > a_dict[sensors[j]] else 0
             for i in range(len(sensors))
             for j in range(i + 1, len(sensors))],
            dtype=np.uint8)

    # ratio bits — new information not computed in the Averaging cell
    geo = np.exp(np.mean([np.log(v) for v in a_dict.values()]))
    ratio = []
    for s in sensors:
        log_r   = np.log2(a_dict[s] / geo)
        clipped = np.clip((log_r + 1.0) / 2.0, 0.0, 1.0 - 1e-9)
        q       = int(clipped * (2 ** n_quant))
        ratio.extend([(q >> b) & 1 for b in range(n_quant - 1, -1, -1)])

    return np.concatenate([rank, np.array(ratio, np.uint8)])


class RSFuzzyExtractor:
    """
    Code-offset fuzzy extractor using Reed-Solomon over GF(2^8).

    n_puf_bytes : total bytes in the PUF fingerprint
    n_ecc_bytes : RS error-correction bytes;
                  corrects up to n_ecc_bytes // 2 bit flips per read
    key_bytes   : output key length in bytes (16 = 128-bit key)
    """
    def __init__(self, n_puf_bytes: int,
                 n_ecc_bytes: int = 2,
                 key_bytes:   int = 16):
        if n_puf_bytes <= n_ecc_bytes:
            raise ValueError(
                f"n_puf_bytes ({n_puf_bytes}) must exceed n_ecc_bytes ({n_ecc_bytes}). "
                f"Increase N_QUANT or decrease N_ECC_BYTES.")
        self.n_puf  = n_puf_bytes
        self.n_ecc  = n_ecc_bytes
        self.n_data = n_puf_bytes - n_ecc_bytes
        self.n_key  = key_bytes
        self.rs     = reedsolo.RSCodec(n_ecc_bytes)

    @property
    def max_byte_errors(self):
        return self.n_ecc // 2

    def gen(self, puf_bits):
        """Enrollment. Returns (key, helper). Key is secret, helper is public."""
        puf_bytes = bits_to_bytes(puf_bits)
        if len(puf_bytes) != self.n_puf:
            raise ValueError(f"PUF gave {len(puf_bytes)} bytes, expected {self.n_puf}")
        secret   = os.urandom(self.n_data)
        codeword = bytes(self.rs.encode(secret))
        helper   = bytes(p ^ c for p, c in zip(puf_bytes, codeword))
        key      = hashlib.sha256(secret).digest()[:self.n_key]
        return key, helper

    def rep(self, put_bits_noisy, helper: bytes):
        """Verification. Returns key bytes on success, None if uncorrectable."""
        puf_bytes = bits_to_bytes(puf_bits_noisy)
        noisy_cw  = bytes(p ^ h for p, h in zip(puf_bytes, helper))
        try:
            decoded, _, _ = self.rs.decode(noisy_cw)
            return hashlib.sha256(bytes(decoded)).digest()[:self.n_key]
        except reedsolo.ReedSolomonError:
            return None


print("\u2713 PUF Toolbox ready.")

## Cell 2 — Enrollment
Run after the Averaging cell. Generates the PUF fingerprint, derives a key, and saves the helper data to disk.

In [ ]:
ENROLL_FILE     = Path("puf_enrollment.json")
N_ENROLL_TRIALS = 1000  # trial windows to average; must be < NUM_TRIALS
N_QUANT         = 4     # ratio bits per sensor (2–4 recommended)
N_ECC_BYTES     = 2     # RS repair bytes; corrects up to N_ECC_BYTES//2 bit flips

###

if 'a_hat' not in dir() or 'NUM_TRIALS' not in dir():
    raise RuntimeError("Run the Averaging cell first.")
if 'modal_bitstring' not in dir():
    raise RuntimeError("modal_bitstring not found. Run the Averaging cell first.")
if NUM_TRIALS < N_ENROLL_TRIALS:
    print(f"WARNING: only {NUM_TRIALS} trials available, "
          f"using all for enrollment.")

# step 1: average amplitudes over enrollment window
n_use    = min(N_ENROLL_TRIALS, NUM_TRIALS)
a_enroll = {s: np.mean([a_hat[g][s] for g in range(n_use)]) for s in SENSORS}

# step 2: extract fingerprint bits
# rank bits come directly from modal_bitstring (already computed in Averaging cell)
# ratio bits are computed fresh from the averaged amplitudes
puf_bits_enroll = extract_puf_bits(a_enroll, SENSORS,
                                    n_quant=N_QUANT,
                                    rank_bits=modal_bitstring)
n_bits      = len(puf_bits_enroll)
n_puf_bytes = len(bits_to_bytes(puf_bits_enroll))

print(f"PUF fingerprint : {n_bits} bits \u2192 {n_puf_bytes} bytes")
print(f"  rank bits     : {len(SENSORS)*(len(SENSORS)-1)//2} "
      f"(reused from Averaging cell, {math.factorial(len(SENSORS))} valid orderings)")
print(f"  ratio bits    : {len(SENSORS) * N_QUANT} "
      f"({N_QUANT} bits \u00d7 {len(SENSORS)} sensors)")
print(f"  bit pattern   : {''.join(map(str, puf_bits_enroll))}")

# step 3: build extractor and generate key + helper
fe = RSFuzzyExtractor(n_puf_bytes=n_puf_bytes,
                      n_ecc_bytes=N_ECC_BYTES,
                      key_bytes=16)
print(f"RS correction capacity : {fe.max_byte_errors} bit flip(s) per read")

key_enroll, helper = fe.gen(puf_bits_enroll)

# step 4: save helper data (NOT the key)
record = {
    "helper"          : helper.hex(),
    "n_puf_bytes"     : n_puf_bytes,
    "n_ecc_bytes"     : N_ECC_BYTES,
    "n_quant"         : N_QUANT,
    "sensors"         : SENSORS,
    "Fs"              : float(Fs),
    "carrier_hz"      : float(CARRIER_FREQ),
    "n_enroll_trials" : n_use,
}
ENROLL_FILE.write_text(json.dumps(record, indent=2))

print(f"\n\u2713 Enrollment complete. Helper saved to {ENROLL_FILE.resolve()}")
print(f"  Key (SECRET, in memory only): {key_enroll[:4].hex()}...{key_enroll[-4:].hex()}")
print(f"  Never save the full key \u2014 regenerate it via verification when needed.")

## Cell 3 — Verification
Run every time you want to authenticate the device. Requires: Cell 1 (Toolbox), Averaging cell, and Cell 2 (Enrollment).

In [ ]:
ENROLL_FILE     = Path("puf_enrollment.json")
N_VERIFY_TRIALS = 1000  # trial windows to average for verification read

###

if 'a_hat' not in dir() or 'NUM_TRIALS' not in dir():
    raise RuntimeError("Run the Averaging cell first.")
if not ENROLL_FILE.exists():
    raise RuntimeError(f"{ENROLL_FILE} not found. Run Enrollment first.")

# load enrollment record
record      = json.loads(ENROLL_FILE.read_text())
helper      = bytes.fromhex(record["helper"])
n_puf_bytes = record["n_puf_bytes"]
n_ecc_bytes = record["n_ecc_bytes"]
N_QUANT_V   = record["n_quant"]
sensors_v   = record["sensors"]
fe          = RSFuzzyExtractor(n_puf_bytes, n_ecc_bytes, key_bytes=16)

# use a non-overlapping window so this is a genuinely fresh read
offset = record.get("n_enroll_trials", 1000)
n_use  = min(N_VERIFY_TRIALS, NUM_TRIALS - offset)
if n_use <= 0:
    raise RuntimeError(
        f"Not enough trials for verification. "
        f"Enrollment used {offset}, total available {NUM_TRIALS}. "
        f"Increase NUM_SAMPLES_TO_SAVE in the Averaging cell.")

a_verify        = {s: np.mean([a_hat[g][s] for g in range(offset, offset + n_use)])
                   for s in sensors_v}
# rank bits reused from Averaging cell modal_bitstring
puf_bits_verify = extract_puf_bits(a_verify, sensors_v,
                                    n_quant=N_QUANT_V,
                                    rank_bits=modal_bitstring)

# diagnostic: bit-level comparison vs enrollment
if 'a_enroll' in dir():
    puf_bits_ref = extract_puf_bits(a_enroll, sensors_v,
                                     n_quant=N_QUANT_V,
                                     rank_bits=modal_bitstring)
    diff = puf_bits_ref ^ puf_bits_verify
    hd   = int(diff.sum())
    print(f"Enroll : {''.join(map(str, puf_bits_ref))}")
    print(f"Verify : {''.join(map(str, puf_bits_verify))}")
    print(f"Diffs  : {''.join('X' if b else '.' for b in diff)}")
    print(f"Hamming distance : {hd}/{len(puf_bits_ref)} bits "
          f"\u2014 RS capacity {fe.max_byte_errors} bit(s)")

# attempt key recovery
key_verify = fe.rep(puf_bits_verify, helper)

print(f"\n{'='*60}")
if key_verify is None:
    print("\u2717  VERIFICATION FAILED \u2014 too many bit errors for RS to correct.")
    print(f"   \u2192 Increase N_ECC_BYTES (currently {n_ecc_bytes}) and re-enroll")
    print(f"   \u2192 Or increase N_VERIFY_TRIALS to reduce noise")
elif 'key_enroll' in dir() and key_verify != key_enroll:
    print("\u2717  KEY MISMATCH \u2014 RS decoded but key differs. Please report as a bug.")
else:
    print("\u2713  VERIFICATION PASSED")
print(f"{'='*60}")

## Cell 4 — Reliability Sweep
Tests reliability across many independent windows. Requires: Cell 1 (Toolbox), Averaging cell, and Cell 2 (Enrollment).

In [ ]:
if 'a_hat' not in dir() or 'NUM_TRIALS' not in dir():
    raise RuntimeError("Run the Averaging cell first.")
if 'a_enroll' not in dir() or 'fe' not in dir() or 'helper' not in dir():
    raise RuntimeError("Run Enrollment (Cell 2) first.")

max_windows = NUM_TRIALS // WINDOW_SIZE
N_WINDOWS   = min(N_WINDOWS, max_windows)

puf_bits_ref = extract_puf_bits(a_enroll, SENSORS,
                                  n_quant=N_QUANT,
                                  rank_bits=modal_bitstring)

hds        = []
passed     = []
bit_errors = np.zeros(len(puf_bits_ref), dtype=int)

print(f"Sweep: {N_WINDOWS} windows \u00d7 {WINDOW_SIZE} trials  |  "
      f"RS capacity: {fe.max_byte_errors} bit flip(s)")
print(f"{'\u2500'*60}")

for w in range(N_WINDOWS):
    start  = w * WINDOW_SIZE
    end    = start + WINDOW_SIZE
    a_w    = {s: np.mean([a_hat[g][s] for g in range(start, end)]) for s in SENSORS}
    bits_w = extract_puf_bits(a_w, SENSORS,
                               n_quant=N_QUANT,
                               rank_bits=modal_bitstring)
    diff   = puf_bits_ref ^ bits_w
    hd     = int(diff.sum())
    hds.append(hd)
    bit_errors += diff
    ok     = fe.rep(bits_w, helper) is not None
    passed.append(ok)
    print(f"  Window {w+1:3d}: HD={hd:2d}  "
          f"[{'PASS' if ok else 'FAIL'}]  "
          + ''.join('X' if b else '.' for b in diff))

n_tested = len(hds)
n_passed = sum(passed)
n_failed = n_tested - n_passed
max_hd   = max(hds)

print(f"\n{'='*60}")
print(f"Passed : {n_passed}/{n_tested} ({100*n_passed/n_tested:.1f}%)")
print(f"HD     : mean={np.mean(hds):.2f}  min={min(hds)}  max={max_hd}")

unstable = np.where(bit_errors > 0)[0]
n_rank   = len(SENSORS) * (len(SENSORS) - 1) // 2
if len(unstable):
    print(f"\nUnstable bits:")
    for idx in unstable:
        kind = "rank" if idx < n_rank else "ratio"
        print(f"  bit[{idx:2d}] ({kind}): flipped "
              f"{bit_errors[idx]}/{n_tested} times "
              f"({100*bit_errors[idx]/n_tested:.1f}%)")

print(f"\nRECOMMENDATION:")
if n_failed == 0 and max_hd == 0:
    print(f"  Perfectly stable. N_ECC_BYTES={N_ECC_BYTES} is more than enough.")
elif n_failed == 0:
    print(f"  All passed. Max errors ({max_hd}) within RS capacity ({fe.max_byte_errors}).")
else:
    print(f"  {n_failed} window(s) failed.")
    print(f"  \u2192 Re-enroll with N_ECC_BYTES = {max_hd * 2} in Cell 2")
print(f"{'='*60}")